# Model

In [ ]:
import requests
import torchvision.models as models
from PIL import Image
from transformers import RobertaTokenizer, RobertaModel, CLIPProcessor, CLIPModel

## Image Encoder

### ResNet

ResNet introduced residual connections, they allow to train networks with an unseen number of layers (up to 1000). ResNet won the 2015 ILSVRC & COCO competition, one important milestone in deep computer vision.

Deep Residual Learning for Image Recognition (He et al., 2015)

Paper: <https://arxiv.org/abs/1512.03385>

![ResNet Block](https://d2l.ai/_images/resnet-block.svg)

In [ ]:
resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
resnet

### Vision Transformers (ViT)

Vision Transformer (ViT) is a transformer adapted for computer vision tasks. An image is split into smaller fixed-sized patches which are treated as a sequence of tokens, similar to words for NLP tasks. ViT requires less resources to pretrain compared to convolutional architectures and its performance on large datasets can be transferred to smaller downstream tasks.

An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale (Dosovitskiy et al., 2020)

Paper: <https://arxiv.org/abs/2010.11929>

![Vision Transformer](https://d2l.ai/_images/vit.svg)

In [ ]:
vit = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
vit

## Text Encoder

### RoBERTa

RoBERTa improves BERT with new pretraining objectives, demonstrating BERT was undertrained and training design is important. The pretraining objectives include dynamic masking, sentence packing, larger batches and a byte-level BPE tokenizer.

RoBERTa: A Robustly Optimized BERT Pretraining Approach (Liu et al., 2019)

Paper: <https://arxiv.org/abs/1907.11692>

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("FacebookAI/roberta-base")
tokenizer

In [ ]:
roberta = RobertaModel.from_pretrained("FacebookAI/roberta-base")

## CLIP

CLIP (Contrastive Language-Image Pre-Training) is a neural network trained on a variety of (image, text) pairs. It can be instructed in natural language to predict the most relevant text snippet, given an image, without directly optimizing for the task, similarly to the zero-shot capabilities of GPT-2 and 3. We found CLIP matches the performance of the original ResNet50 on ImageNet “zero-shot” without using any of the original 1.28M labeled examples, overcoming several major challenges in computer vision.

1. GitHub: <https://github.com/openai/CLIP>

2. Blog: <https://openai.com/blog/clip/>

3. Paper: <https://arxiv.org/abs/2103.00020>

![CLIP](https://github.com/openai/CLIP/raw/main/CLIP.png)

In [ ]:
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")
processor

In [ ]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16")
model

In [ ]:
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)
image

In [ ]:
inputs = processor(
    text=["a photo of a cat", "a photo of a dog"],
    images=image,
    return_tensors="pt",
    padding=True,
)
outputs = model(**inputs)
# The image-text similarity score
logits_per_image = outputs.logits_per_image
# Take the softmax to get the label probabilities
probs = logits_per_image.softmax(dim=1)
probs